## 1. Imports and file paths

The path helper below lets the notebook run either in the project folder with a `data/` directory or directly from the folder where the uploaded CSV files are stored.

In [4]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [5]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path("/mnt/data")

OUT_DIR = BASE_DIR / "cleaned_outputs"
OUT_DIR.mkdir(exist_ok=True)

print("Using DATA_DIR:", DATA_DIR.resolve())
print("Saving cleaned outputs to:", OUT_DIR.resolve())

Using DATA_DIR: C:\Users\anaso\Documents\MSSE (need to organize)\Spring 26\277B ML\Chem-277B-MLP\patient_datafile\Checkpoint3_Step1\data
Saving cleaned outputs to: C:\Users\anaso\Documents\MSSE (need to organize)\Spring 26\277B ML\Chem-277B-MLP\patient_datafile\Checkpoint3_Step1\cleaned_outputs


In [8]:
def read_csv_file(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)

protein = read_csv_file("egfr_protein.csv")
rna = read_csv_file("egfr_rna.csv")
full = read_csv_file("egfr_full_merged.csv")
mutation = read_csv_file("egfr_mutation_updated.csv") if (DATA_DIR / "egfr_mutation_updated.csv").exists() else read_csv_file("egfr_mutation.csv")
phospho = read_csv_file("egfr_phospho.csv")
binding_df = read_csv_file("egfr_ligand_binding.csv")
smiles_df = read_csv_file("ligand_smiles_features.csv")
patient_ligand_raw = read_csv_file("patient_ligand.csv")

for name, df in {
    "protein": protein,
    "rna": rna,
    "full": full,
    "mutation": mutation,
    "phospho": phospho,
    "binding_df": binding_df,
    "smiles_df": smiles_df,
    "patient_ligand_raw": patient_ligand_raw,
}.items():
    print(f"{name:20s}", df.shape)

protein              (203, 2)
rna                  (197, 4)
full                 (207, 4)
mutation             (70, 10)
phospho              (1620, 5)
binding_df           (29, 6)
smiles_df            (3, 11)
patient_ligand_raw   (350, 22)


## 3. Standardize patient IDs and cohort labels

Cleaning technique used in the notebooks:

- rename `PATIENT_ID` to `patient_id`
- remove CPTAC tumor/normal suffixes like `.T` and `.N`
- strip whitespace
- add `cohort` and `batch_domain` labels so CPTAC and TCGA sources are not treated as directly interchangeable

In [9]:
def clean_patient_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.N$", "", regex=True)
        .str.replace(r"\.T$", "", regex=True)
    )

for df in [protein, rna, full, phospho, mutation]:
    if "PATIENT_ID" in df.columns and "patient_id" not in df.columns:
        df.rename(columns={"PATIENT_ID": "patient_id"}, inplace=True)
    if "Patient_ID" in df.columns and "patient_id" not in df.columns:
        df.rename(columns={"Patient_ID": "patient_id"}, inplace=True)
    if "patient_id" in df.columns:
        df["patient_id"] = clean_patient_id(df["patient_id"])

for df in [protein, rna, full, phospho]:
    df["cohort"] = "CPTAC"
    df["batch_domain"] = "CPTAC"

mutation["cohort"] = "TCGA"
mutation["batch_domain"] = "TCGA"
mutation["gene"] = "EGFR"

print(full[["patient_id", "cohort", "batch_domain"]].head())
print(mutation[["patient_id", "cohort", "batch_domain", "gene"]].head())

  patient_id cohort batch_domain
0  C3L-00001  CPTAC        CPTAC
1  C3L-00009  CPTAC        CPTAC
2  C3L-00080  CPTAC        CPTAC
3  C3L-00083  CPTAC        CPTAC
4  C3L-00093  CPTAC        CPTAC
        patient_id cohort batch_domain  gene
0  TCGA-05-4382-01   TCGA         TCGA  EGFR
1  TCGA-05-4402-01   TCGA         TCGA  EGFR
2  TCGA-05-4410-01   TCGA         TCGA  EGFR
3  TCGA-05-5423-01   TCGA         TCGA  EGFR
4  TCGA-17-Z026-01   TCGA         TCGA  EGFR


## 4. CPTAC expression features


The expression table keeps EGFR RNA, EGFR protein, and activity mean. Duplicate patient IDs are collapsed by averaging numeric expression values.

In [10]:
expression_cols = ["patient_id", "EGFR_PROTEIN", "EGFR_RNA", "EGFR_activity_mean", "cohort", "batch_domain"]
expression_protein_features = full[[c for c in expression_cols if c in full.columns]].copy()

numeric_expr_cols = [c for c in ["EGFR_PROTEIN", "EGFR_RNA", "EGFR_activity_mean"] if c in expression_protein_features.columns]
for col in numeric_expr_cols:
    expression_protein_features[col] = pd.to_numeric(expression_protein_features[col], errors="coerce")

expression_protein_features = (
    expression_protein_features
    .groupby("patient_id", as_index=False)
    .agg({**{c: "mean" for c in numeric_expr_cols}, "cohort": "first", "batch_domain": "first"})
)

print("expression_protein_features:", expression_protein_features.shape)
print("duplicate patient IDs:", expression_protein_features["patient_id"].duplicated().sum())
expression_protein_features.head()

expression_protein_features: (106, 6)
duplicate patient IDs: 0


,patient_id,EGFR_PROTEIN,EGFR_RNA,EGFR_activity_mean,cohort,batch_domain
0,C3L-00001,26.919190,14.500,1.0,CPTAC,CPTAC
1,C3L-00009,25.188724,12.090,1.0,CPTAC,CPTAC
2,C3L-00080,25.203323,12.530,1.0,CPTAC,CPTAC
3,C3L-00083,25.336752,11.725,1.0,CPTAC,CPTAC
4,C3L-00093,24.736987,12.750,1.0,CPTAC,CPTAC


## 5. CPTAC phosphosite reshaping and signaling score

Cleaning technique used:

- filter to the EGFR phosphosites used in the project
- pivot from long format to one row per patient
- rename columns as `phospho_Y####`
- calculate an average EGFR phosphorylation/expression score across available sites

In [12]:
target_sites = ["Y1172", "Y1092", "Y1069", "Y1110", "Y1016"]

phospho_by_site = phospho[phospho["EGFR_binding_site"].isin(target_sites)].copy()
phospho_by_site["patient_id"] = clean_patient_id(phospho_by_site["patient_id"])
phospho_by_site["EGFR_phospho_value"] = pd.to_numeric(phospho_by_site["EGFR_phospho_value"], errors="coerce")

phospho_model_data = (
    phospho_by_site
    .pivot_table(index="patient_id", columns="EGFR_binding_site", values="EGFR_phospho_value", aggfunc="mean")
    .reset_index()
)
phospho_model_data.columns = ["patient_id"] + [f"phospho_{c}" for c in phospho_model_data.columns[1:]]

egfr_expression_score = (
    phospho_by_site
    .groupby("patient_id", as_index=False)["EGFR_phospho_value"]
    .mean()
    .rename(columns={"EGFR_phospho_value": "EGFR_expression_score"})
)

print("phospho_model_data:", phospho_model_data.shape)
print("egfr_expression_score:", egfr_expression_score.shape)
phospho_model_data.head()

phospho_model_data: (64, 6)
egfr_expression_score: (64, 2)


,patient_id,phospho_Y1016,phospho_Y1069,phospho_Y1092,phospho_Y1110,phospho_Y1172
0,C3L-00001,NaN,NaN,NaN,NaN,20.075879
1,C3L-00009,NaN,NaN,16.931486,NaN,18.767056
2,C3L-00080,NaN,NaN,NaN,NaN,18.026248
3,C3L-00140,NaN,NaN,17.530325,NaN,18.448596
4,C3L-00263,NaN,NaN,NaN,NaN,18.784880


## 6. Build the CPTAC model table

This is the cleaned CPTAC biological feature table used before ligand fusion. Missing phosphosite values are kept at this stage so the notebook can show what was missing before model-level imputation.

In [14]:
cptac_model_df = (
    expression_protein_features
    .merge(phospho_model_data, on="patient_id", how="left", validate="one_to_one")
    .merge(egfr_expression_score, on="patient_id", how="left", validate="one_to_one")
)

print("cptac_model_df:", cptac_model_df.shape)
print("duplicate patient IDs:", cptac_model_df["patient_id"].duplicated().sum())
cptac_model_df.to_csv(OUT_DIR / "cptac_model_df.csv", index=False)
cptac_model_df.head()

cptac_model_df: (106, 12)
duplicate patient IDs: 0


,patient_id,EGFR_PROTEIN,EGFR_RNA,EGFR_activity_mean,cohort,batch_domain,phospho_Y1016,phospho_Y1069,phospho_Y1092,phospho_Y1110,phospho_Y1172,EGFR_expression_score
0,C3L-00001,26.919190,14.500,1.0,CPTAC,CPTAC,NaN,NaN,NaN,NaN,20.075879,20.075879
1,C3L-00009,25.188724,12.090,1.0,CPTAC,CPTAC,NaN,NaN,16.931486,NaN,18.767056,17.849271
2,C3L-00080,25.203323,12.530,1.0,CPTAC,CPTAC,NaN,NaN,NaN,NaN,18.026248,18.026248
3,C3L-00083,25.336752,11.725,1.0,CPTAC,CPTAC,NaN,NaN,NaN,NaN,NaN,NaN
4,C3L-00093,24.736987,12.750,1.0,CPTAC,CPTAC,NaN,NaN,NaN,NaN,NaN,NaN


## 7. TCGA mutation annotation and mutation-feature cleaning

Cleaning techniques used:

- strip mutation strings
- identify EGFR hotspot labels
- classify mutation types
- create binary mutation features for modeling
- mark compound mutations using mutation counts

In [15]:
mutation["mutation"] = mutation["mutation"].astype(str).str.strip()
mutation["EGFR_type"] = mutation.get("EGFR_type", "unknown").astype(str).str.strip()

def label_egfr_hotspot(mutation_value: str) -> str:
    m = str(mutation_value).upper().strip()
    if (
        "EXON 19" in m or "19DEL" in m or "DEL19" in m
        or re.search(r"E\d+_A\d+DEL", m)
        or re.search(r"L\d+_A\d+DEL", m)
        or re.search(r"L\d+_T\d+DEL", m)
        or "DELINS" in m
    ):
        return "exon19del"
    elif "L858R" in m:
        return "L858R"
    elif "T790M" in m:
        return "T790M"
    elif "C797S" in m:
        return "C797S"
    elif "G719" in m:
        return "G719X"
    elif "L861Q" in m:
        return "L861Q"
    elif "S768I" in m:
        return "S768I"
    elif "EXON 20" in m or "INS" in m or "DUP" in m:
        return "exon20_alteration"
    else:
        return "other"

def classify_mutation(row) -> str:
    text = f"{row.get('mutation', '')} {row.get('EGFR_type', '')}".upper()
    if "MISSENSE" in text:
        return "missense"
    elif "NONSENSE" in text or "STOP" in text:
        return "nonsense"
    elif "FRAMESHIFT" in text:
        return "frameshift"
    elif "SPLICE" in text:
        return "splice"
    elif "AMP" in text or "AMPLIFICATION" in text:
        return "amplification"
    elif "DEL" in text or "DELETION" in text:
        return "deletion"
    elif "SYNONYMOUS" in text or "SILENT" in text:
        return "synonymous"
    else:
        return "other"

mutation["egfr_hotspot_label"] = mutation["mutation"].apply(label_egfr_hotspot)
mutation["mutation_class"] = mutation.apply(classify_mutation, axis=1)

nonsynonymous_classes = {"missense", "nonsense", "frameshift", "splice", "amplification", "deletion"}
mutation["is_nonsynonymous"] = mutation["mutation_class"].isin(nonsynonymous_classes).astype(int)
mutation["has_exon19del"] = mutation["egfr_hotspot_label"].eq("exon19del").astype(int)
mutation["has_L858R"] = mutation["egfr_hotspot_label"].eq("L858R").astype(int)
mutation["has_L861Q"] = mutation["egfr_hotspot_label"].eq("L861Q").astype(int)
mutation["has_G719X"] = mutation["egfr_hotspot_label"].eq("G719X").astype(int)
mutation["has_exon20_alteration"] = mutation["egfr_hotspot_label"].eq("exon20_alteration").astype(int)
mutation["has_other_mutation"] = mutation["egfr_hotspot_label"].eq("other").astype(int)
mutation["mutation_count"] = mutation["mutation"].str.split().str.len().fillna(0).astype(int)
mutation["is_compound_mutation"] = (mutation["mutation_count"] > 1).astype(int)
mutation["is_egfr_hotspot"] = mutation["egfr_hotspot_label"].ne("other").astype(int)

mutation_features = mutation[[
    "patient_id", "mutation", "EGFR_type", "cohort", "batch_domain", "gene",
    "egfr_hotspot_label", "mutation_class", "is_nonsynonymous",
    "has_exon19del", "has_L858R", "has_L861Q", "has_G719X",
    "has_exon20_alteration", "has_other_mutation", "mutation_count",
    "is_compound_mutation", "is_egfr_hotspot"
]].copy()

mutation_features.to_csv(OUT_DIR / "mutation_features_cleaned.csv", index=False)
print("mutation_features:", mutation_features.shape)
print(mutation_features["egfr_hotspot_label"].value_counts(dropna=False))
mutation_features.head()

mutation_features: (70, 18)
egfr_hotspot_label
exon19del            24
L858R                23
other                14
L861Q                 3
exon20_alteration     3
G719X                 3
Name: count, dtype: int64


,patient_id,mutation,EGFR_type,cohort,batch_domain,gene,egfr_hotspot_label,mutation_class,is_nonsynonymous,has_exon19del,has_L858R,has_L861Q,has_G719X,has_exon20_alteration,has_other_mutation,mutation_count,is_compound_mutation,is_egfr_hotspot
0,TCGA-05-4382-01,R222L E545Q,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,2,1,0
1,TCGA-05-4402-01,T751_I759delinsN I759N,Exon19,TCGA,TCGA,EGFR,exon19del,deletion,1,1,0,0,0,0,0,2,1,1
2,TCGA-05-4410-01,R377S,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,1,0,0
3,TCGA-05-5423-01,L833F L861Q,Other,TCGA,TCGA,EGFR,L861Q,other,0,0,0,1,0,0,0,2,1,1
4,TCGA-17-Z026-01,G721V,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,1,0,0


## 8. Master sample table

This table documents which patient IDs are represented in CPTAC expression/protein/phospho data versus TCGA mutation data. This was important because the TCGA and CPTAC patient IDs do not fully overlap, so the project kept the cohort/batch information explicit.

In [16]:
all_patient_ids = pd.Series(
    pd.concat([
        expression_protein_features["patient_id"],
        phospho["patient_id"],
        mutation_features["patient_id"]
    ]).dropna().unique(),
    name="patient_id"
)

master_samples = pd.DataFrame(all_patient_ids)
master_samples["has_rna"] = master_samples["patient_id"].isin(expression_protein_features["patient_id"]).astype(int)
master_samples["has_protein"] = master_samples["patient_id"].isin(expression_protein_features["patient_id"]).astype(int)
master_samples["has_phospho"] = master_samples["patient_id"].isin(phospho["patient_id"]).astype(int)
master_samples["has_mutation"] = master_samples["patient_id"].isin(mutation_features["patient_id"]).astype(int)
master_samples["cohort"] = np.where(master_samples["patient_id"].astype(str).str.startswith("C3L-"), "CPTAC", "TCGA")
master_samples["batch_domain"] = master_samples["cohort"]

master_samples.to_csv(OUT_DIR / "master_samples.csv", index=False)
print("master_samples:", master_samples.shape)
master_samples.head()

master_samples: (176, 7)


,patient_id,has_rna,has_protein,has_phospho,has_mutation,cohort,batch_domain
0,C3L-00001,1,1,1,0,CPTAC,CPTAC
1,C3L-00009,1,1,1,0,CPTAC,CPTAC
2,C3L-00080,1,1,1,0,CPTAC,CPTAC
3,C3L-00083,1,1,1,0,CPTAC,CPTAC
4,C3L-00093,1,1,1,0,CPTAC,CPTAC


## 9. ChEMBL ligand binding and SMILES cleaning

Cleaning techniques used:

- force binding values to numeric
- remove missing or non-positive binding values
- summarize duplicate ligand measurements with median binding values
- remove duplicate SMILES rows by ligand
- merge binding summaries with chemical descriptors

In [18]:
binding_df["standard_value"] = pd.to_numeric(binding_df["standard_value"], errors="coerce")
binding_df["p_binding"] = pd.to_numeric(binding_df["p_binding"], errors="coerce")

binding_clean = binding_df.dropna(subset=["ligand", "protein", "binding_type", "standard_value", "p_binding"]).copy()
binding_clean = binding_clean[binding_clean["standard_value"] > 0].copy()

ligand_binding_summary = (
    binding_clean
    .groupby(["ligand", "protein", "binding_type"], as_index=False)
    .agg(
        median_binding_nM=("standard_value", "median"),
        median_p_binding=("p_binding", "median"),
        n_measurements=("standard_value", "count")
    )
)
ligand_binding_summary["standard_units"] = "nM"

smiles_clean = smiles_df.drop_duplicates(subset=["ligand"]).copy()
ligand_master = ligand_binding_summary.merge(smiles_clean, on="ligand", how="left", validate="many_to_one")
ligand_master.to_csv(OUT_DIR / "ligand_master.csv", index=False)

print("ligand_binding_summary:", ligand_binding_summary.shape)
print("ligand_master:", ligand_master.shape)
ligand_master.head()

ligand_binding_summary: (3, 7)
ligand_master: (3, 17)


,ligand,protein,binding_type,median_binding_nM,median_p_binding,n_measurements,standard_units,pref_name,canonical_smiles,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,RingCount,HeavyAtomCount
0,CHEMBL1079742,CHEMBL203,IC50,42.500,7.489405,4,nM,ERLOTINIB HYDROCHLORIDE,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl,429.904,3.82690,74.73,1,7,10,3,30
1,CHEMBL941,CHEMBL203,IC50,50000.055,6.979304,2,nM,IMATINIB,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37
2,CHEMBL941,CHEMBL203,Kd,10000.000,5.000000,23,nM,IMATINIB,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37


## 10. CPTAC + ChEMBL fusion table

From a previous file`checkpont_3_part1.ipynb`, the project created a cross-product between the CPTAC patient feature table and the EGFR ligand table. Then the table was cleaned for model use:

- drop missing patient/ligand IDs
- impute numeric columns with medians
- impute categorical columns with `unknown`
- standardize text case for drug names and binding types
- create interaction features such as `mutation_binding_context` and `protein_ligand_pair`

In [19]:
patient_mut = (
    mutation_features[["patient_id", "mutation", "egfr_hotspot_label", "is_egfr_hotspot"]]
    .drop_duplicates(subset=["patient_id"])
)

cptac_with_mut = cptac_model_df.drop_duplicates(subset=["patient_id"]).merge(
    patient_mut, on="patient_id", how="left", validate="one_to_one"
)
cptac_with_mut["mutation"] = cptac_with_mut["mutation"].fillna("unknown")
cptac_with_mut["egfr_hotspot_label"] = cptac_with_mut["egfr_hotspot_label"].fillna("other")
cptac_with_mut["is_egfr_hotspot"] = cptac_with_mut["is_egfr_hotspot"].fillna(0)

cptac_with_mut["key"] = 1
ligand_master["key"] = 1
model_df = cptac_with_mut.merge(ligand_master, on="key", how="inner").drop(columns=["key"])

model_df = model_df.dropna(subset=["patient_id", "ligand"]).copy()
numeric_cols = model_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in model_df.select_dtypes(exclude=["number"]).columns if c != "patient_id"]

for col in numeric_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())
for col in categorical_cols:
    model_df[col] = model_df[col].fillna("unknown")

if "pref_name" in model_df.columns:
    model_df["pref_name"] = model_df["pref_name"].astype(str).str.strip().str.lower()
model_df["binding_type"] = model_df["binding_type"].astype(str).str.strip().str.upper()
model_df["egfr_hotspot_label"] = model_df["egfr_hotspot_label"].astype(str).str.strip()

model_df["mutation_binding_context"] = model_df["egfr_hotspot_label"].astype(str) + "_" + model_df["binding_type"].astype(str)
model_df["protein_ligand_pair"] = model_df["protein"].astype(str) + "_" + model_df["ligand"].astype(str)

model_df.to_csv(OUT_DIR / "model_df_clean.csv", index=False)
print("model_df:", model_df.shape)
model_df.head()

model_df: (318, 34)


,patient_id,EGFR_PROTEIN,EGFR_RNA,EGFR_activity_mean,cohort,batch_domain,phospho_Y1016,phospho_Y1069,phospho_Y1092,phospho_Y1110,phospho_Y1172,EGFR_expression_score,mutation,egfr_hotspot_label,is_egfr_hotspot,ligand,protein,binding_type,median_binding_nM,median_p_binding,n_measurements,standard_units,pref_name,canonical_smiles,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,RingCount,HeavyAtomCount,mutation_binding_context,protein_ligand_pair
0,C3L-00001,26.919190,14.50,1.0,CPTAC,CPTAC,13.072192,12.604645,17.767915,10.583122,20.075879,20.075879,unknown,other,0.0,CHEMBL1079742,CHEMBL203,IC50,42.500,7.489405,4,nM,erlotinib hydrochloride,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl,429.904,3.82690,74.73,1,7,10,3,30,other_IC50,CHEMBL203_CHEMBL1079742
1,C3L-00001,26.919190,14.50,1.0,CPTAC,CPTAC,13.072192,12.604645,17.767915,10.583122,20.075879,20.075879,unknown,other,0.0,CHEMBL941,CHEMBL203,IC50,50000.055,6.979304,2,nM,imatinib,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37,other_IC50,CHEMBL203_CHEMBL941
2,C3L-00001,26.919190,14.50,1.0,CPTAC,CPTAC,13.072192,12.604645,17.767915,10.583122,20.075879,20.075879,unknown,other,0.0,CHEMBL941,CHEMBL203,KD,10000.000,5.000000,23,nM,imatinib,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37,other_KD,CHEMBL203_CHEMBL941
3,C3L-00009,25.188724,12.09,1.0,CPTAC,CPTAC,13.072192,12.604645,16.931486,10.583122,18.767056,17.849271,unknown,other,0.0,CHEMBL1079742,CHEMBL203,IC50,42.500,7.489405,4,nM,erlotinib hydrochloride,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl,429.904,3.82690,74.73,1,7,10,3,30,other_IC50,CHEMBL203_CHEMBL1079742
4,C3L-00009,25.188724,12.09,1.0,CPTAC,CPTAC,13.072192,12.604645,16.931486,10.583122,18.767056,17.849271,unknown,other,0.0,CHEMBL941,CHEMBL203,IC50,50000.055,6.979304,2,nM,imatinib,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37,other_IC50,CHEMBL203_CHEMBL941


## 11. One-hot encoded ML-ready table

This reproduces the notebook step where categorical model features were encoded with `pd.get_dummies`. Patient ID, ligand ID, protein ID, and SMILES are kept in the table for traceability, but they should usually be removed from `X` before fitting a model unless intentionally encoded.

In [20]:
categorical_to_encode = [
    "cohort", "batch_domain", "mutation", "egfr_hotspot_label", "binding_type", "pref_name",
    "mutation_binding_context", "protein_ligand_pair"
]
existing_catg = [c for c in categorical_to_encode if c in model_df.columns]

ml_ready_df = pd.get_dummies(model_df, columns=existing_catg, drop_first=False)
ml_ready_df.to_csv(OUT_DIR / "model_df_ml_ready.csv", index=False)

print("ml_ready_df:", ml_ready_df.shape)
ml_ready_df.head()

ml_ready_df: (318, 38)


,patient_id,EGFR_PROTEIN,EGFR_RNA,EGFR_activity_mean,phospho_Y1016,phospho_Y1069,phospho_Y1092,phospho_Y1110,phospho_Y1172,EGFR_expression_score,is_egfr_hotspot,ligand,protein,median_binding_nM,median_p_binding,n_measurements,standard_units,canonical_smiles,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,RingCount,HeavyAtomCount,cohort_CPTAC,batch_domain_CPTAC,mutation_unknown,egfr_hotspot_label_other,binding_type_IC50,binding_type_KD,pref_name_erlotinib hydrochloride,pref_name_imatinib,mutation_binding_context_other_IC50,mutation_binding_context_other_KD,protein_ligand_pair_CHEMBL203_CHEMBL1079742,protein_ligand_pair_CHEMBL203_CHEMBL941
0,C3L-00001,26.919190,14.50,1.0,13.072192,12.604645,17.767915,10.583122,20.075879,20.075879,0.0,CHEMBL1079742,CHEMBL203,42.500,7.489405,4,nM,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl,429.904,3.82690,74.73,1,7,10,3,30,True,True,True,True,True,False,True,False,True,False,True,False
1,C3L-00001,26.919190,14.50,1.0,13.072192,12.604645,17.767915,10.583122,20.075879,20.075879,0.0,CHEMBL941,CHEMBL203,50000.055,6.979304,2,nM,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37,True,True,True,True,True,False,False,True,True,False,False,True
2,C3L-00001,26.919190,14.50,1.0,13.072192,12.604645,17.767915,10.583122,20.075879,20.075879,0.0,CHEMBL941,CHEMBL203,10000.000,5.000000,23,nM,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37,True,True,True,True,False,True,False,True,False,True,False,True
3,C3L-00009,25.188724,12.09,1.0,13.072192,12.604645,16.931486,10.583122,18.767056,17.849271,0.0,CHEMBL1079742,CHEMBL203,42.500,7.489405,4,nM,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl,429.904,3.82690,74.73,1,7,10,3,30,True,True,True,True,True,False,True,False,True,False,True,False
4,C3L-00009,25.188724,12.09,1.0,13.072192,12.604645,16.931486,10.583122,18.767056,17.849271,0.0,CHEMBL941,CHEMBL203,50000.055,6.979304,2,nM,Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1,493.615,4.59032,86.28,2,7,7,5,37,True,True,True,True,True,False,False,True,True,False,False,True


## 12. Clean `patient_ligand.csv` for the LightGBM modeling step

Your newer workflow used `patient_ligand.csv` as the patient–ligand bridge for LightGBM modeling. The uploaded file has the mutation columns plus four ligand columns. Because the ligand columns were saved without clean names, this cell repairs them into:

- `ligand_id`
- `ligand_name`
- `ligand_alias_or_description`
- `canonical_smiles`

Then it applies the same basic cleaning logic used elsewhere: strip IDs, standardize text, fill missing categorical ligand values, and save a clean version.

In [21]:
patient_ligand = patient_ligand_raw.copy()

# Repair unnamed / accidentally named ligand columns after the known mutation-feature columns.
base_cols = [
    "patient_id", "mutation", "EGFR_type", "cohort", "batch_domain", "gene",
    "egfr_hotspot_label", "mutation_class", "is_nonsynonymous", "has_exon19del",
    "has_L858R", "has_L861Q", "has_G719X", "has_exon20_alteration",
    "has_other_mutation", "mutation_count", "is_compound_mutation", "is_egfr_hotspot"
]
extra_cols = [c for c in patient_ligand.columns if c not in base_cols]
rename_extra = {}
if len(extra_cols) >= 1:
    rename_extra[extra_cols[0]] = "ligand_id"
if len(extra_cols) >= 2:
    rename_extra[extra_cols[1]] = "ligand_name"
if len(extra_cols) >= 3:
    rename_extra[extra_cols[2]] = "ligand_alias_or_description"
if len(extra_cols) >= 4:
    rename_extra[extra_cols[3]] = "canonical_smiles"
patient_ligand = patient_ligand.rename(columns=rename_extra)

patient_ligand["patient_id"] = clean_patient_id(patient_ligand["patient_id"])
for col in ["mutation", "EGFR_type", "cohort", "batch_domain", "gene", "egfr_hotspot_label", "mutation_class"]:
    if col in patient_ligand.columns:
        patient_ligand[col] = patient_ligand[col].astype(str).str.strip()

for col in ["ligand_id", "ligand_name", "ligand_alias_or_description", "canonical_smiles"]:
    if col in patient_ligand.columns:
        patient_ligand[col] = patient_ligand[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan})
        patient_ligand[col] = patient_ligand[col].fillna("unknown")

binary_cols = [
    "is_nonsynonymous", "has_exon19del", "has_L858R", "has_L861Q", "has_G719X",
    "has_exon20_alteration", "has_other_mutation", "is_compound_mutation", "is_egfr_hotspot"
]
for col in binary_cols:
    if col in patient_ligand.columns:
        patient_ligand[col] = pd.to_numeric(patient_ligand[col], errors="coerce").fillna(0).astype(int)
if "mutation_count" in patient_ligand.columns:
    patient_ligand["mutation_count"] = pd.to_numeric(patient_ligand["mutation_count"], errors="coerce").fillna(0).astype(int)

patient_ligand = patient_ligand.drop_duplicates().reset_index(drop=True)
patient_ligand.to_csv(OUT_DIR / "patient_ligand_clean.csv", index=False)

print("patient_ligand_clean:", patient_ligand.shape)
print(patient_ligand.columns.tolist())
patient_ligand.head()

patient_ligand_clean: (350, 22)
['patient_id', 'mutation', 'EGFR_type', 'cohort', 'batch_domain', 'gene', 'egfr_hotspot_label', 'mutation_class', 'is_nonsynonymous', 'has_exon19del', 'has_L858R', 'has_L861Q', 'has_G719X', 'has_exon20_alteration', 'has_other_mutation', 'mutation_count', 'is_compound_mutation', 'is_egfr_hotspot', 'ligand_id', 'ligand_name', 'ligand_alias_or_description', 'canonical_smiles']


,patient_id,mutation,EGFR_type,cohort,batch_domain,gene,egfr_hotspot_label,mutation_class,is_nonsynonymous,has_exon19del,has_L858R,has_L861Q,has_G719X,has_exon20_alteration,has_other_mutation,mutation_count,is_compound_mutation,is_egfr_hotspot,ligand_id,ligand_name,ligand_alias_or_description,canonical_smiles
0,TCGA-05-4382-01,R222L E545Q,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,2,1,0,260,5-{[(5-{[(2S)-1-carboxy-3-oxopropan-2-yl]carbamoyl}-4-methylthiophen-2-yl)methyl]sulfamoyl}-2-hydroxybenzoic acid::(...,Cc1cc(CNS(=O)(=O)c2ccc(O)c(c2)C(O)=O)sc1C(=O)N[C@@H](CC(O)=O)C=O |r|,unknown
1,TCGA-05-4382-01,R222L E545Q,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,2,1,0,279,6-[(4-Hydroxy-3-methyl-benzenesulfonylamino)methyl]-N-[1-(7-methoxy-benzooxazole-2-carbonyl)propyl]nicotinamide::5-{...,3-benzoxazol-2-yl)-1-oxopropan-2-yl]carbamoyl}pyridin-2-yl)methyl]sulfamoyl}-2-hydroxybenzoic acid::Pyridine Scaffol...,COc1cccc2nc(oc12)C(=O)[C@H](CC(O)=O)NC(=O)c1ccc(CNS(=O)(=O)c2ccc(O)c(c2)C(O)=O)nc1 |r|
2,TCGA-05-4382-01,R222L E545Q,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,2,1,0,13534,VX680::N-[4-[[4-(4-methylpiperazino)-6-[(5-methyl-1H-pyrazol-3-yl)amino]pyrimidin-2-yl]thio]phenyl]cyclopropanecarbo...,CN1CCN(CC1)c1cc(Nc2cc(C)n[nH]2)nc(Sc2ccc(NC(=O)C3CC3)cc2)n1,unknown
3,TCGA-05-4382-01,R222L E545Q,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,2,1,0,13645,4-amino-3-[2-(2,4-dichlorophenyl)ethoxy]-N-{[1-(pyridin-4-yl)piperidin-4-yl]methyl}benzamide::3-Oxybenzamide 31,Nc1ccc(cc1OCCc1ccc(Cl)cc1Cl)C(=O)NCC1CCN(CC1)c1ccncc1
4,TCGA-05-4382-01,R222L E545Q,Other,TCGA,TCGA,EGFR,other,other,0,0,0,0,0,0,1,2,1,0,24828,N-[2-(1H-indol-3-yl)ethyl][(naphthalen-2-ylmethyl)sulfanyl]carbothioamide::Brassinin derivative,16,S=C(NCCc1c[nH]c2ccccc12)SCc1ccc2ccccc2c1


## 13. LightGBM-ready feature matrix from `patient_ligand_clean`

This cell prepares the exact kind of matrix used for LightGBM: mutation indicators and ligand identifiers/descriptions are cleaned, categorical variables are one-hot encoded, and non-feature ID columns are excluded from `X`.

If your final LightGBM target column is added later, place it in `target_candidates` or rename it to one of the listed names.

In [23]:
target_candidates = [
    "predicted_binding_affinity", "binding_affinity", "p_binding", "median_p_binding",
    "log_binding_affinity", "median_binding_nM", "target"
]
target_col = next((c for c in target_candidates if c in patient_ligand.columns), None)

exclude_cols = ["patient_id"]
if target_col:
    exclude_cols.append(target_col)

feature_df = patient_ligand.drop(columns=[c for c in exclude_cols if c in patient_ligand.columns]).copy()

# Do not directly use raw long SMILES strings unless you intentionally want sparse string identifiers.
# Keeping it here as a categorical feature mirrors the patient-ligand notebook logic; chemical descriptors can be merged later.
cat_cols = feature_df.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = feature_df.select_dtypes(include=["number", "bool"]).columns.tolist()

for col in num_cols:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="coerce").fillna(feature_df[col].median())
for col in cat_cols:
    feature_df[col] = feature_df[col].fillna("unknown").astype(str)

X_patient_ligand = pd.get_dummies(feature_df, columns=cat_cols, drop_first=False)
X_patient_ligand.to_csv(OUT_DIR / "patient_ligand_lightgbm_features.csv", index=False)

print("Target column detected:", target_col)
print("X_patient_ligand shape:", X_patient_ligand.shape)
X_patient_ligand.head()

Target column detected: None
X_patient_ligand shape: (350, 71)


,is_nonsynonymous,has_exon19del,has_L858R,has_L861Q,has_G719X,has_exon20_alteration,has_other_mutation,mutation_count,is_compound_mutation,is_egfr_hotspot,mutation_A767_V769dup,mutation_D770_N771insGL,mutation_E709_T710delinsD,mutation_E746_A750del,mutation_E866K,mutation_G627*,mutation_G719A,mutation_G719A S768I,mutation_G719C S768I,mutation_G721V,mutation_H773dup,mutation_I91V R1052I,mutation_K754E L747_E749del,mutation_K754I G901V Q432H,mutation_K754_I759del L833V,mutation_L62R L858R,mutation_L747_A750delinsP,mutation_L747_T751del,mutation_L833F L861Q,mutation_L858R,mutation_L858R T790M,mutation_L861Q,mutation_L861Q D1083Efs*11 L387M,mutation_L907M,mutation_Q486*,mutation_R222L E545Q,mutation_R377S,mutation_S921R,mutation_T751_I759delinsN I759N,mutation_V300M,mutation_X210_splice,EGFR_type_Exon19,EGFR_type_L858R,EGFR_type_Other,cohort_TCGA,batch_domain_TCGA,gene_EGFR,egfr_hotspot_label_L858R,egfr_hotspot_label_other,mutation_class_deletion,mutation_class_other,mutation_class_splice,ligand_id_13534,ligand_id_13645,ligand_id_24828,ligand_id_260,ligand_id_279,ligand_name_4-amino-3-[2-(2,ligand_name_5-{[(5-{[(2S)-1-carboxy-3-oxopropan-2-yl]carbamoyl}-4-methylthiophen-2-yl)methyl]sulfamoyl}-2-hydroxybenzoic acid::(S)-5-{[5-(1-Carboxymethyl-2-oxo-ethylcarbamoyl)-4-methyl-thiophen-2-ylmethyl]sulfamoyl}-2-hydroxy-benzoic Acid::Heterocyclic deriv. 69a::Inhibitor 69a,ligand_name_6-[(4-Hydroxy-3-methyl-benzenesulfonylamino)methyl]-N-[1-(7-methoxy-benzooxazole-2-carbonyl)propyl]nicotinamide::5-{[(5-{[(2S)-3-carboxy-1-(7-methoxy-1,ligand_name_N-[2-(1H-indol-3-yl)ethyl][(naphthalen-2-ylmethyl)sulfanyl]carbothioamide::Brassinin derivative,ligand_name_VX680::N-[4-[[4-(4-methylpiperazino)-6-[(5-methyl-1H-pyrazol-3-yl)amino]pyrimidin-2-yl]thio]phenyl]cyclopropanecarboxamide::cyclopropane carboxylic acid {4-[4-(4-methyl-piperazin-1-yl)-6-(5-methyl-2H-pyrazol-3-ylamino)-pyrimidin-2ylsulphanyl]-phenyl}-amide::CHEMBL572878::N-[4-({4-[(3-methyl-1H-pyrazol-5-yl)amino]-6-(4-methylpiperazin-1-yl)pyrimidin-2-yl}sulfanyl)phenyl]cyclopropanecarboxamide::VX-680,ligand_alias_or_description_16,ligand_alias_or_description_3-benzoxazol-2-yl)-1-oxopropan-2-yl]carbamoyl}pyridin-2-yl)methyl]sulfamoyl}-2-hydroxybenzoic acid::Pyridine Scaffold 61::Inhibitor 61,ligand_alias_or_description_4-dichlorophenyl)ethoxy]-N-{[1-(pyridin-4-yl)piperidin-4-yl]methyl}benzamide::3-Oxybenzamide 31,ligand_alias_or_description_CN1CCN(CC1)c1cc(Nc2cc(C)n[nH]2)nc(Sc2ccc(NC(=O)C3CC3)cc2)n1,ligand_alias_or_description_Cc1cc(CNS(=O)(=O)c2ccc(O)c(c2)C(O)=O)sc1C(=O)N[C@@H](CC(O)=O)C=O |r|,canonical_smiles_COc1cccc2nc(oc12)C(=O)[C@H](CC(O)=O)NC(=O)c1ccc(CNS(=O)(=O)c2ccc(O)c(c2)C(O)=O)nc1 |r|,canonical_smiles_Nc1ccc(cc1OCCc1ccc(Cl)cc1Cl)C(=O)NCC1CCN(CC1)c1ccncc1,canonical_smiles_S=C(NCCc1c[nH]c2ccccc12)SCc1ccc2ccccc2c1,canonical_smiles_unknown
0,0,0,0,0,0,0,1,2,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,True,True,False,True,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,True
1,0,0,0,0,0,0,1,2,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,True,True,False,True,False,True,False,False,False,False,False,True,False,False,True,False,False,False,True,False,False,False,True,False,False,False
2,0,0,0,0,0,0,1,2,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,True,True,True,False,True,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True
3,0,0,0,0,0,0,1,2,1,0,False,False,Fal

## 15. Summary of cleaning techniques combined

This notebook combines the data-cleaning methods used across your checkpoint notebooks:

1. **Patient ID normalization:** standardized `PATIENT_ID`/`patient_id`, removed `.N` and `.T`, and stripped whitespace.
2. **Cohort tracking:** assigned `CPTAC` or `TCGA` to avoid pretending the datasets came from one identical source.
3. **Expression cleaning:** selected EGFR RNA/protein/activity columns and collapsed duplicates by patient.
4. **Phosphosite reshaping:** filtered EGFR tyrosine sites, pivoted long-to-wide, and created an average EGFR signaling score.
5. **Mutation engineering:** classified mutation type, hotspot label, binary hotspot flags, mutation count, and compound mutation status.
6. **Master sample tracking:** marked which patients had RNA, protein, phospho, and mutation data.
7. **ChEMBL binding cleaning:** converted values to numeric, dropped invalid rows, summarized repeated measurements by median, and merged SMILES descriptors.
8. **Model fusion:** merged patient biology with ligand binding/chemistry, imputed missing values, cleaned categorical text, and created interaction features.